# EDA & Preprocessing - AG News Classification

This notebook performs exploratory data analysis on the **AG News** dataset (4-class news classification) and demonstrates preprocessing pipelines for both TF-IDF baseline and DistilBERT transformer models.

**Sections:**
1. Imports & Setup
2. Load Data
3. Dataset Overview
4. Class Distribution
5. Text Length Analysis
6. Word Frequency Analysis
7. Text Overlap Analysis
8. N-gram Analysis
9. Tokenization with DistilBERT
10. TF-IDF Feature Analysis
11. Summary of Key Findings

## 1. Imports & Setup

In [ ]:
import sys
import os
import warnings

# Add project root to path so we can import from src/
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import string

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# Plot styling
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100
warnings.filterwarnings("ignore")

print("Setup complete.")

## 2. Load Data

We use the project's `data_loader` module which loads AG News from Hugging Face datasets, falling back to synthetic data if the library isn't available.

In [ ]:
from src.data_loader import load_ag_news, LABEL_NAMES

train_df, test_df = load_ag_news()

# Map numeric labels to human-readable names
label_names = LABEL_NAMES  # {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
train_df["label_name"] = train_df["label"].map(label_names)
test_df["label_name"] = test_df["label"].map(label_names)

print(f"Train samples: {len(train_df):,}")
print(f"Test samples:  {len(test_df):,}")
print(f"Label mapping: {label_names}")

## 3. Dataset Overview

In [ ]:
# Basic shape and info
print("=" * 50)
print("TRAINING SET")
print("=" * 50)
print(f"Shape: {train_df.shape}")
print(f"Columns: {list(train_df.columns)}")
print(f"\nLabel distribution:")
print(train_df["label_name"].value_counts().to_string())
print(f"\nNull values: {train_df.isnull().sum().sum()}")
print(f"Duplicate texts: {train_df['text'].duplicated().sum()}")

print("\n" + "=" * 50)
print("TEST SET")
print("=" * 50)
print(f"Shape: {test_df.shape}")
print(f"\nLabel distribution:")
print(test_df["label_name"].value_counts().to_string())

In [ ]:
# Sample rows from each class
print("Sample texts per class:\n")
for label_id, name in label_names.items():
    sample = train_df[train_df["label"] == label_id].iloc[0]["text"]
    print(f"[{name}] {sample[:120]}...")
    print()

In [ ]:
# Show first few rows as a table
train_df.head(10)

## 4. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training set class distribution
class_order = ["World", "Sports", "Business", "Sci/Tech"]
colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]

train_counts = train_df["label_name"].value_counts().reindex(class_order)
axes[0].bar(class_order, train_counts.values, color=colors, edgecolor="white", linewidth=1.2)
axes[0].set_title("Training Set - Class Distribution", fontweight="bold")
axes[0].set_ylabel("Number of Samples")
for i, v in enumerate(train_counts.values):
    axes[0].text(i, v + 200, f"{v:,}", ha="center", fontweight="bold")

# Test set class distribution
test_counts = test_df["label_name"].value_counts().reindex(class_order)
axes[1].bar(class_order, test_counts.values, color=colors, edgecolor="white", linewidth=1.2)
axes[1].set_title("Test Set - Class Distribution", fontweight="bold")
axes[1].set_ylabel("Number of Samples")
for i, v in enumerate(test_counts.values):
    axes[1].text(i, v + 10, f"{v:,}", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

# Check balance ratio
train_ratio = train_counts.max() / train_counts.min()
print(f"Train class imbalance ratio (max/min): {train_ratio:.2f}")
print("The dataset is well-balanced across all 4 classes." if train_ratio < 1.1 else "Some class imbalance detected.")

## 5. Text Length Analysis

In [ ]:
# Compute text statistics
train_df["char_length"] = train_df["text"].str.len()
train_df["word_count"] = train_df["text"].str.split().str.len()
train_df["avg_word_length"] = train_df["text"].apply(
    lambda x: np.mean([len(w) for w in x.split()]) if x.split() else 0
)

# Summary statistics per class
length_stats = train_df.groupby("label_name").agg(
    mean_chars=("char_length", "mean"),
    median_chars=("char_length", "median"),
    std_chars=("char_length", "std"),
    mean_words=("word_count", "mean"),
    median_words=("word_count", "median"),
    mean_word_len=("avg_word_length", "mean"),
).round(1)

print("Text Length Statistics per Class:")
length_stats

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram of character lengths
for name, color in zip(class_order, colors):
    subset = train_df[train_df["label_name"] == name]["char_length"]
    axes[0].hist(subset, bins=50, alpha=0.5, label=name, color=color, density=True)
axes[0].set_title("Character Length Distribution", fontweight="bold")
axes[0].set_xlabel("Character Length")
axes[0].set_ylabel("Density")
axes[0].legend()

# Histogram of word counts
for name, color in zip(class_order, colors):
    subset = train_df[train_df["label_name"] == name]["word_count"]
    axes[1].hist(subset, bins=50, alpha=0.5, label=name, color=color, density=True)
axes[1].set_title("Word Count Distribution", fontweight="bold")
axes[1].set_xlabel("Word Count")
axes[1].set_ylabel("Density")
axes[1].legend()

# Boxplot of word counts per class
box_data = [train_df[train_df["label_name"] == name]["word_count"].values for name in class_order]
bp = axes[2].boxplot(box_data, labels=class_order, patch_artist=True)
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[2].set_title("Word Count by Class", fontweight="bold")
axes[2].set_ylabel("Word Count")

plt.tight_layout()
plt.show()

## 6. Word Frequency Analysis

In [ ]:
# Define stopwords (common English) to filter out uninformative words
STOPWORDS = {
    "the", "a", "an", "is", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "do", "does", "did", "will", "would", "could",
    "should", "may", "might", "shall", "can", "need", "dare", "ought",
    "to", "of", "in", "for", "on", "with", "at", "by", "from", "as",
    "into", "through", "during", "before", "after", "above", "below",
    "between", "out", "off", "over", "under", "again", "further",
    "then", "once", "here", "there", "when", "where", "why", "how",
    "all", "both", "each", "few", "more", "most", "other", "some",
    "such", "no", "nor", "not", "only", "own", "same", "so", "than",
    "too", "very", "just", "because", "but", "and", "or", "if", "while",
    "about", "up", "it", "its", "his", "her", "their", "our", "my",
    "he", "she", "they", "we", "you", "i", "me", "him", "them", "us",
    "this", "that", "these", "those", "what", "which", "who", "whom",
    "said", "also", "new", "one", "two", "first", "last", "s", "t",
    "it", "amp", "quot", "lt", "gt",  # HTML artifacts common in AG News
}


def get_top_words(texts, n=20):
    """Extract top n words after removing stopwords and punctuation."""
    all_words = []
    for text in texts:
        words = re.findall(r'\b[a-z]+\b', text.lower())
        all_words.extend([w for w in words if w not in STOPWORDS and len(w) > 2])
    return Counter(all_words).most_common(n)


# Get top words for each class
top_words_per_class = {}
for label_id, name in label_names.items():
    texts = train_df[train_df["label"] == label_id]["text"]
    top_words_per_class[name] = get_top_words(texts, n=20)

# Display
for name, words in top_words_per_class.items():
    print(f"\n--- {name} (Top 10) ---")
    for word, count in words[:10]:
        print(f"  {word:20s} {count:,}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, (name, color) in zip(axes.flat, zip(class_order, colors)):
    words, counts = zip(*top_words_per_class[name])
    ax.barh(range(len(words)), counts, color=color, alpha=0.8)
    ax.set_yticks(range(len(words)))
    ax.set_yticklabels(words)
    ax.invert_yaxis()
    ax.set_title(f"Top 20 Words - {name}", fontweight="bold")
    ax.set_xlabel("Frequency")

plt.tight_layout()
plt.show()

## 7. Text Overlap Analysis

How distinguishable are the classes based on their vocabulary? We measure the ratio of unique words that appear predominantly in one class.

In [ ]:
# Build vocabulary per class
class_vocabs = {}
for label_id, name in label_names.items():
    texts = train_df[train_df["label"] == label_id]["text"]
    words = set()
    for text in texts:
        words.update(re.findall(r'\b[a-z]+\b', text.lower()))
    class_vocabs[name] = words

# Unique words per class (not shared with any other class)
all_classes = list(class_vocabs.keys())
print("Vocabulary Analysis:\n")
print(f"{'Class':<12} {'Total Vocab':>12} {'Unique Words':>14} {'Unique %':>10}")
print("-" * 52)

unique_counts = {}
for name in all_classes:
    other_words = set()
    for other_name in all_classes:
        if other_name != name:
            other_words |= class_vocabs[other_name]
    unique = class_vocabs[name] - other_words
    unique_counts[name] = len(unique)
    total = len(class_vocabs[name])
    pct = 100 * len(unique) / total if total > 0 else 0
    print(f"{name:<12} {total:>12,} {len(unique):>14,} {pct:>9.1f}%")

# Shared vocabulary across all classes
shared_all = class_vocabs["World"] & class_vocabs["Sports"] & class_vocabs["Business"] & class_vocabs["Sci/Tech"]
print(f"\nWords shared by ALL classes: {len(shared_all):,}")

In [ ]:
# Pairwise vocabulary overlap (Jaccard similarity)
overlap_matrix = pd.DataFrame(index=class_order, columns=class_order, dtype=float)

for c1 in class_order:
    for c2 in class_order:
        intersection = len(class_vocabs[c1] & class_vocabs[c2])
        union = len(class_vocabs[c1] | class_vocabs[c2])
        overlap_matrix.loc[c1, c2] = intersection / union if union > 0 else 0

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    overlap_matrix.astype(float),
    annot=True, fmt=".3f", cmap="YlOrRd",
    square=True, ax=ax, vmin=0, vmax=1,
    linewidths=0.5
)
ax.set_title("Pairwise Vocabulary Overlap (Jaccard Similarity)", fontweight="bold")
plt.tight_layout()
plt.show()

print("Higher overlap means classes share more vocabulary and may be harder to distinguish.")

## 8. N-gram Analysis

Top bigrams per class reveal characteristic phrases.

In [ ]:
def get_top_ngrams(texts, n=2, top_k=15):
    """Extract top-k n-grams from a collection of texts."""
    vec = CountVectorizer(ngram_range=(n, n), stop_words="english", max_features=10000)
    try:
        X = vec.fit_transform(texts)
        # Sum counts across all documents
        counts = X.sum(axis=0).A1
        vocab = vec.get_feature_names_out()
        top_idx = counts.argsort()[::-1][:top_k]
        return [(vocab[i], int(counts[i])) for i in top_idx]
    except Exception:
        return []


# Top bigrams per class
bigrams_per_class = {}
for label_id, name in label_names.items():
    texts = train_df[train_df["label"] == label_id]["text"].tolist()
    bigrams_per_class[name] = get_top_ngrams(texts, n=2, top_k=15)

for name in class_order:
    print(f"\n--- {name} (Top 10 Bigrams) ---")
    for gram, count in bigrams_per_class[name][:10]:
        print(f"  {gram:30s} {count:,}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, (name, color) in zip(axes.flat, zip(class_order, colors)):
    bigrams = bigrams_per_class[name][:15]
    if bigrams:
        grams, counts = zip(*bigrams)
        ax.barh(range(len(grams)), counts, color=color, alpha=0.8)
        ax.set_yticks(range(len(grams)))
        ax.set_yticklabels(grams)
        ax.invert_yaxis()
    ax.set_title(f"Top Bigrams - {name}", fontweight="bold")
    ax.set_xlabel("Frequency")

plt.tight_layout()
plt.show()

## 9. Preprocessing: DistilBERT Tokenization

Demonstrate how the DistilBERT tokenizer converts raw text into model inputs: token IDs, attention masks, and subword tokens.

In [ ]:
try:
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    HAS_TOKENIZER = True
    print(f"Tokenizer: distilbert-base-uncased")
    print(f"Vocabulary size: {tokenizer.vocab_size:,}")
    print(f"Max model input length: {tokenizer.model_max_length:,}")
except ImportError:
    HAS_TOKENIZER = False
    print("transformers not installed. Showing conceptual example below.")

In [ ]:
# Tokenization example
sample_texts = [
    "Apple unveils new MacBook with M4 chip at developer conference",
    "US women's soccer team wins Olympic gold in thrilling penalty shootout",
    "Federal Reserve raises interest rates by 25 basis points",
    "NASA's James Webb telescope discovers high-redshift galaxy candidates",
]

if HAS_TOKENIZER:
    print("Tokenization Examples:\n")
    for text in sample_texts:
        encoded = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=32,
            return_tensors="pt",
        )
        tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
        non_pad_tokens = [t for t in tokens if t != "[PAD]"]

        print(f"Text:       {text}")
        print(f"Tokens:     {non_pad_tokens}")
        print(f"Token IDs:  {encoded['input_ids'][0][:len(non_pad_tokens)].tolist()}")
        print(f"Attn mask:  {encoded['attention_mask'][0][:len(non_pad_tokens)].tolist()}")
        print(f"Decoded:    {tokenizer.decode(encoded['input_ids'][0], skip_special_tokens=True)}")
        print(f"Num tokens: {len(non_pad_tokens)} (incl. [CLS] and [SEP])")
        print()
else:
    print("""Conceptual tokenization flow:
    1. Text -> WordPiece subword tokens
    2. Add special tokens: [CLS] text [SEP] [PAD]...
    3. Convert to token IDs (integers)
    4. Create attention mask (1 for real, 0 for padding)
    5. Truncate/pad to max_length (128)
    """)

In [ ]:
# Token length distribution — how many tokens does the dataset need?
if HAS_TOKENIZER:
    # Sample 5000 texts for speed
    sample = train_df["text"].sample(n=min(5000, len(train_df)), random_state=42)
    token_lengths = sample.apply(lambda x: len(tokenizer.encode(x, truncation=False)))

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(token_lengths, bins=60, color="#4C72B0", alpha=0.8, edgecolor="white")
    ax.axvline(128, color="red", linestyle="--", linewidth=2, label="max_length=128")
    ax.set_title("DistilBERT Token Length Distribution (sample of 5K texts)", fontweight="bold")
    ax.set_xlabel("Number of Tokens")
    ax.set_ylabel("Count")
    ax.legend()
    plt.tight_layout()
    plt.show()

    pct_under_128 = (token_lengths <= 128).mean() * 100
    print(f"Texts fitting within 128 tokens: {pct_under_128:.1f}%")
    print(f"Median token length: {token_lengths.median():.0f}")
    print(f"95th percentile: {token_lengths.quantile(0.95):.0f}")
else:
    print("Tokenizer not available. Typical AG News articles fit within 128 tokens.")

## 10. TF-IDF Feature Analysis

Fit a TF-IDF vectorizer and examine the most discriminative features per class.

In [ ]:
from src.data_loader import create_tfidf_features

tfidf_data = create_tfidf_features(train_df, test_df, max_features=50000)
vectorizer = tfidf_data["vectorizer"]
X_train_tfidf = tfidf_data["X_train"]

print(f"TF-IDF matrix shape: {X_train_tfidf.shape}")
print(f"Sparsity: {1 - X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1]):.4%}")

In [ ]:
# Top TF-IDF features per class (by mean TF-IDF score within each class)
feature_names = vectorizer.get_feature_names_out()

print("Top 15 TF-IDF Features per Class:\n")
for label_id, name in label_names.items():
    mask = tfidf_data["y_train"] == label_id
    class_tfidf = X_train_tfidf[mask].mean(axis=0).A1  # Average TF-IDF per feature
    top_idx = class_tfidf.argsort()[::-1][:15]
    top_features = [(feature_names[i], class_tfidf[i]) for i in top_idx]

    print(f"--- {name} ---")
    for feat, score in top_features:
        print(f"  {feat:25s} {score:.4f}")
    print()

In [ ]:
# Visualize top TF-IDF features per class
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, (label_id, name), color in zip(axes.flat, label_names.items(), colors):
    mask = tfidf_data["y_train"] == label_id
    class_tfidf = X_train_tfidf[mask].mean(axis=0).A1
    top_idx = class_tfidf.argsort()[::-1][:15]
    top_feats = [feature_names[i] for i in top_idx]
    top_scores = [class_tfidf[i] for i in top_idx]

    ax.barh(range(len(top_feats)), top_scores, color=color, alpha=0.8)
    ax.set_yticks(range(len(top_feats)))
    ax.set_yticklabels(top_feats)
    ax.invert_yaxis()
    ax.set_title(f"Top TF-IDF Features - {name}", fontweight="bold")
    ax.set_xlabel("Mean TF-IDF Score")

plt.tight_layout()
plt.show()

## 11. Summary of Key Findings

### Dataset Characteristics
- **Size**: 120,000 training / 7,600 test samples across 4 balanced classes
- **Classes**: World, Sports, Business, Sci/Tech — evenly distributed
- **Text length**: Articles are relatively short (typically 30-60 words), fitting well within DistilBERT's 128-token limit

### Class Distinguishability
- **Sports** has the most distinctive vocabulary (team names, game terminology)
- **Sci/Tech** is also fairly distinct (technology companies, research terms)
- **World** and **Business** share the most vocabulary overlap, which may lead to more confusion between these classes

### Preprocessing Insights
- TF-IDF with unigrams + bigrams captures discriminative phrases (e.g., "World Cup", "stock market", "interest rate")
- DistilBERT tokenization handles subwords well — most texts fit within 128 tokens
- Minimal preprocessing needed for the transformer (no stemming/lemmatization required)

### Implications for Modeling
- The balanced dataset means no special handling for class imbalance
- TF-IDF baselines should perform well given the distinctive vocabularies
- Transformers should excel at World vs Business distinction thanks to contextual embeddings
- `max_length=128` is sufficient for the vast majority of samples